In [ ]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

# import logging
# logging.basicConfig(
#     level=logging.DEBUG,
#     format="%(asctime)s %(levelname)s %(name)s: %(message)s",
#     handlers=[
#         logging.FileHandler("curation.log"),
#         logging.StreamHandler(),  # keep console output too
#     ],
#     force=True,
# )

# Download data


In [ ]:
noncurated_path = "../non_curated/h5ad/frangieh_2021_raw.h5ad"
download_file(
    url="https://exampledata.scverse.org/pertpy/frangieh_2021_raw.h5ad",
    dest_path=noncurated_path
)

# Initialise the dataset object

In [ ]:
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    noncurated_path=noncurated_path
)

cur_data.load_data()

Filter the data on MOI == 1. This keeps only cells targeted with 1 guideRNA.

In [ ]:
cur_data.adata = cur_data.adata[cur_data.adata.obs['MOI'] == "1"]

# OBS slot curation

### Rename sgRNA to perturbation_name

In [ ]:
cur_data.rename_columns(slot = 'obs', name_dict = {'sgRNA': 'perturbation_name'})
cur_data.adata.obs

### Show unique perturbations

In [ ]:
cur_data.show_unique(slot = 'obs', column = 'perturbation_name')

### Add guide RNA information

In [ ]:
download_file(
    url="https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-021-00779-1/MediaObjects/41588_2021_779_MOESM3_ESM.xlsx",
    dest_path="../supplementary/frangieh_2021_supp.xlsx"
)

guide_info_df = pd.read_excel("../supplementary/frangieh_2021_supp.xlsx", sheet_name="Supplementary Table 1", skiprows=2)
guide_info_df = guide_info_df.rename(columns = {'Guide Name':'perturbation_name','sgRNA Sequence':'guide_sequence'})
guide_info_df

In [ ]:
cur_data.adata.obs = cur_data.adata.obs.merge(guide_info_df, on='perturbation_name', how='left')
cur_data.adata.obs

### Standardise perturbation targets

Two types of controls are used:
1. Non-targeting controls: `NO_SITE_*`
2. Controls targeting intergenic regions: `ONE_NON-GENE_SITE_*`

In [ ]:
cur_data.adata.obs['target'] = (cur_data.adata.obs['perturbation_name']
    .str.replace('NO_SITE', 'control_nontargeting') # rename nontargeting control guides
    .str.replace('ONE_NON-GENE_SITE', 'control_intergenic') # rename intergenic control guides
    .str.replace(r"_\d+", "", regex=True) # remove guide number suffix
    .fillna('control_casonly')
)

In [ ]:
cur_data.standardize_genes(
    slot='obs',
    input_column='target',
    input_column_type='gene_symbol',
    multiple_entries=False,
    # multiple_entries_sep='|'
)

### Add `perturbed_target_number` column

In [ ]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_symbol',
    count_column_name='perturbed_target_number',
    sep='|'
)

### Encode chromosomes as integers

In [ ]:
cur_data.chromosome_encoding()

### Add treatment information

In [ ]:
cur_data.adata.obs['treatment_label'] = cur_data.adata.obs['condition'].replace({
    'Control': 'untreated control',
    'IFNγ': 'Interferon gamma',
    'Co-culture': 'Interferon gamma|Tumor Infiltrating Lymphocyte'
})

cur_data.adata.obs['treatment_id'] = cur_data.adata.obs['condition'].replace({
    'Control': 'NCIT:C184729',
    'IFNγ': 'CHEMBL:3286073',
    'Co-culture': 'CHEMBL:3286073|NCIT:C12546'
})

### Add timepoint information

In [ ]:
cur_data.adata.obs['timepoint'] = cur_data.adata.obs['condition'].replace({
    'Control': 'P14DT16H0M0S',
    'IFNγ': 'P14DT16H0M0S',
    'Co-culture': 'P17DT16H0M0S'
})

### Add metadata

In [ ]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        #----- dataset -----#
        "dataset_id": cur_data.dataset_id,
        #----- sample -----#
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        #----- perturbation type -----#
        "perturbation_type_label": "CRISPRn",
        "perturbation_type_id": None,
        #----- data modality -----#
        "data_modality": "Perturb-seq", # different from "method_name_label"; more general term - choice of CRISPR, MAVE and Perturb-seq
        #----- significance -----#
        "significant": None,
        "significance_criteria": None,
        #----- score interpretation -----#
        "score_interpretation": None,
        #----- treatment -----#
        # "treatment_label": None,
        # "treatment_id": None,
        #----- replicate -----#
        "technical_replicate": None,
        "biological_replicate": None,
        #----- model system -----#
        "model_system_label": "primary_cell",
        "model_system_id": None,
        #----- tissue -----#
        "tissue": "skin of body",
        #----- cell line -----#
        "cell_line_label": None,
        "cell_line_id": None,
        #----- cell type -----#
        "cell_type_label": "melanoma cell",
        "cell_type_id": "BTO:0000848",
        #----- disease -----#
        "disease_label": "melanoma",
        "disease_id": "MONDO:0005105",
        #----- timepoint -----#
        # "timepoint": "P7DT0H0M0S",
        #----- species -----#
        "species": "Homo sapiens",
        #----- sex -----#
        "sex_label": None,
        "sex_id": None,
        #----- developmental stage -----#
        "developmental_stage_label": None,
        "developmental_stage_id": None,
        #----- study metadata -----#
        "study_title": "Multimodal pooled Perturb-CITE-seq screens in patient models define mechanisms of cancer immune evasion",
        "study_uri": "https://doi.org/10.1038/s41588-021-00779-1",
        "study_year": 2021,
        #----- authors -----#
        "first_author": "Chris J. Frangieh",
        "last_author": "Benjamin Izar",
        #----- experiment metadata -----#
        "experiment_title": "Perturb-seq CRISPRko screen of primary melanoma cells in untreated, IFNg-stimulated and IFNg-stimulated + co-cultured with autologous tumor-infiltrating lymphocytes to explore cancer cell-intrinsic immune checkpoint inhibitor resistance mechanisms.",
        "experiment_summary": """
            Patient-derived melanoma cells stably expressing Cas9 were engineered by lentiviral transduction of a pooled CROPseq-mKate2 sgRNA library targeting 248 immunotherapy resistance genes. Transduced cells were cultured under antibiotic selection for 14 days, pre-treated with IFN-γ for 16 hours, and subsequently subjected to control, IFN-γ treatment, or co-culture with autologous tumor-infiltrating lymphocytes (TILs) conditions for 48 hours. At day 17, surviving cancer cells were harvested, stained with oligonucleotide-conjugated antibodies, sorted to remove TILs, processed using the Chromium Single Cell 3’ Library and Gel Bead kit v3, and sequenced on an Illumina HiSeq.
        """,
        #----- number of perturbed targets/samples -----#
        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_coord'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],
        #----- library generation type -----#
        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",
        #----- library generation method -----#
        "library_generation_method_id": "EFO:0022876",
        "library_generation_method_label": "SpCas9",
        #----- enzyme and library delivery method -----#
        "enzyme_delivery_method_id": None,
        "enzyme_delivery_method_label": "lentivirus transduction",

        "library_delivery_method_id": None,
        "library_delivery_method_label": "lentivirus transduction",
        #----- enzyme and library integration state -----#
        "enzyme_integration_state_id": None,
        "enzyme_integration_state_label": "random locus integration",

        "library_integration_state_id": None,
        "library_integration_state_label": "random locus integration",
        #----- enzyme and library expression control -----#
        "enzyme_expression_control_id": None,
        "enzyme_expression_control_label": "constitutive transgene expression",

        "library_expression_control_id": None,
        "library_expression_control_label": "constitutive transgene expression",
        #----- library name and URI and manufacturer -----#
        "library_name": "custom",
        "library_uri": None,
        "library_manufacturer": "Izar lab",
        #----- library format -----#
        "library_format_id": None,
        "library_format_label": "pooled",
        #----- library scope -----#
        "library_scope_id": None,
        "library_scope_label": "focused",
        #----- library perturbation type -----#
        "library_perturbation_type_id": None,
        "library_perturbation_type_label": "knockout",
        #----- library additional metadata -----#
        "library_lentiviral_generation": "2",
        "library_grnas_per_target": "3",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()), # for CRISPR/Perturb-seq
        "library_total_variants": None, # for MAVE
        #----- readout dimensionality -----#
        "readout_dimensionality_id": None,
        "readout_dimensionality_label": "high-dimensional assay",
        #---- readout type -----#
        "readout_type_id": None,
        "readout_type_label": "transcriptomic",
        #----- readout technology -----#
        "readout_technology_id": None,
        "readout_technology_label": "single-cell rna-seq",
        #----- method -----#
        "method_name_id": None,
        "method_name_label": "Perturb-CITE-seq", # different from "data_modality"; more specific term - specific name of the technique
        "method_uri": None,
        #----- sequencing library kit -----#
        "sequencing_library_kit_id": None,
        "sequencing_library_kit_label": "10x Genomics Single Cell 3-prime v3",
        #----- sequencing platform -----#
        "sequencing_platform_id": None,
        "sequencing_platform_label": "Illumina HiSeq 2500",
        #----- sequencing strategy -----#
        "sequencing_strategy_id": None,
        "sequencing_strategy_label": "barcode sequencing",
        #----- software used for counts-----#
        "software_counts_id": None,
        "software_counts_label": "CellRanger",
        #----- software used for analysis -----#
        "software_analysis_id": None,
        "software_analysis_label": "MAST",
        #----- reference genome -----#
        "reference_genome_id": None,
        "reference_genome_label": "GRCh38",
        #----- license -----#
        "license_label": "MIT License",
        "license_id": "SWO:9000074",
        #----- external datasets -----#
        "associated_datasets": json.dumps([
            {
                "dataset_accession": "frangieh_2021_raw.h5ad",
                "dataset_uri": "https://scverse-exampledata.s3.eu-west-1.amazonaws.com/pertpy/frangieh_2021_raw.h5ad",
                "dataset_description": "Raw counts - .h5ad file from pertpy",
                "dataset_file_name": "frangieh_2021_raw.h5ad",
            }
        ])
    }
)

### Curate tissue information


In [ ]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

### Match schema column order

In [ ]:
cur_data.match_schema_columns(slot='obs')

### Validate obs metadata

In [ ]:
cur_data.validate_data(slot='obs', verbose=True)

# VAR slot curation

### Standardise genes

In [ ]:
cur_data.adata.var['gene_name'] = cur_data.adata.var.index
cur_data.adata.var

In [ ]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_name",
    input_column_type="gene_symbol",
    remove_version=True,
    multiple_entries=False
)

In [ ]:
cur_data.adata.var

### Validate var metadata

In [ ]:
cur_data.validate_data(slot='var')

# Save the dataset

In [ ]:
cur_data.save_curated_data_h5ad()

In [ ]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

# Upload to BigQuery

In [ ]:
upload_parquet_to_bq(
    parquet_path='../curated/parquet/frangieh_2021_raw_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

# Upload to GC Storage

In [ ]:
!gcloud storage cp ../curated/h5ad/frangieh_2021_raw_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/